# 第16章 什么是 AI、机器学习与科学推断

这个 notebook 用一个极小恒星样本，对照规则 baseline 和一个最小数据驱动分类器，帮助我们区分“会预测”与“会解释”并不是同一件事。

## 学习目标

- 从数据中组织出特征、标签和最小学习任务
- 建立一个可解释的规则 baseline
- 用一个极简数据驱动方法做对照
- 比较规则与数据驱动方法各自学到了什么
- 理解高分不自动等于物理解释

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from collections import Counter, defaultdict
from pathlib import Path

DATA_PATH = Path('../../data/small/stellar_photometry_demo.csv')

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    row['bp_rp'] = float(row['bp_rp'])
    row['parallax_mas'] = float(row['parallax_mas'])
    row['g_mag'] = float(row['g_mag'])
    row['absolute_g_mag'] = row['g_mag'] - 10.0 + 5.0 * math.log10(row['parallax_mas'])

class_counts = Counter(row['stellar_class'] for row in rows)
print('样本数量:', len(rows))
print('类别计数:', dict(class_counts))
print('Python version:', platform.python_version())


In [ ]:
by_class = defaultdict(list)
for row in rows:
    by_class[row['stellar_class']].append(row)

print('按类别的特征范围:')
for stellar_class in sorted(by_class):
    bp_rp_values = [row['bp_rp'] for row in by_class[stellar_class]]
    absolute_mag_values = [row['absolute_g_mag'] for row in by_class[stellar_class]]
    print(
        stellar_class,
        {
            'bp_rp': (round(min(bp_rp_values), 3), round(max(bp_rp_values), 3)),
            'absolute_g_mag': (round(min(absolute_mag_values), 3), round(max(absolute_mag_values), 3)),
        },
    )


## 一个纯规则 baseline

这里故意先不用任何机器学习库，而是直接写出几条物理直觉规则。这样做的意义不是追求最优，而是给后续比较提供一个清楚的起点。

In [ ]:
def classify_star_rule(bp_rp, absolute_g_mag):
    if absolute_g_mag > 10.0:
        return 'white_dwarf'
    if absolute_g_mag < 2.0 and bp_rp > 1.2:
        return 'giant'
    return 'main_sequence'

rule_predictions = []
for row in rows:
    predicted = classify_star_rule(row['bp_rp'], row['absolute_g_mag'])
    rule_predictions.append((row['label'], row['stellar_class'], predicted))

rule_accuracy = sum(truth == predicted for _, truth, predicted in rule_predictions) / len(rule_predictions)
print('rule-based accuracy =', round(rule_accuracy, 3))
print('前五个规则预测:')
for item in rule_predictions[:5]:
    print('  ', item)


## 一个最小数据驱动对照

下面我们用“最近类别中心”的方式做一个极简数据驱动分类器。这里用留一法直觉演示：每次把当前样本拿掉，再用其余样本定义各类中心。

In [ ]:
def feature_vector(row):
    return (row['bp_rp'], row['absolute_g_mag'])

def squared_distance(vec_a, vec_b):
    return sum((a - b) ** 2 for a, b in zip(vec_a, vec_b))

def centroid(rows_subset):
    vectors = [feature_vector(row) for row in rows_subset]
    dims = len(vectors[0])
    return tuple(sum(vector[i] for vector in vectors) / len(vectors) for i in range(dims))

loo_predictions = []
for holdout_index, holdout_row in enumerate(rows):
    grouped = defaultdict(list)
    for index, row in enumerate(rows):
        if index == holdout_index:
            continue
        grouped[row['stellar_class']].append(row)
    centroids = {stellar_class: centroid(group_rows) for stellar_class, group_rows in grouped.items()}
    holdout_vector = feature_vector(holdout_row)
    predicted = min(
        centroids,
        key=lambda stellar_class: squared_distance(holdout_vector, centroids[stellar_class]),
    )
    loo_predictions.append((holdout_row['label'], holdout_row['stellar_class'], predicted))

loo_accuracy = sum(truth == predicted for _, truth, predicted in loo_predictions) / len(loo_predictions)
print('leave-one-out centroid accuracy =', round(loo_accuracy, 3))
print('前五个数据驱动预测:')
for item in loo_predictions[:5]:
    print('  ', item)


In [ ]:
rule_errors = [item for item in rule_predictions if item[1] != item[2]]
loo_errors = [item for item in loo_predictions if item[1] != item[2]]

print('Rule-baseline errors:', rule_errors)
print('Data-driven errors:', loo_errors)

print('只被规则方法分错的样本:')
for label, truth, predicted in rule_errors:
    if (label, truth, predicted) not in loo_errors:
        print('  ', (label, truth, predicted))


## 解释

这个示例说明了两件重要的事。第一，规则 baseline 已经把颜色和绝对星等的物理直觉编码进去了，因此它不是“低级方法”，而是理解问题的起点。第二，数据驱动方法并没有绕开科学知识；它依然依赖特征空间中存在可分结构，只是把边界从人工阈值改成了由样本分布决定。后续真正的机器学习流程，还需要训练/测试切分、评估设计和更细的误差诊断。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'class_counts': dict(class_counts),
    'rule_accuracy': round(rule_accuracy, 3),
    'loo_centroid_accuracy': round(loo_accuracy, 3),
    'rule_error_count': len(rule_errors),
    'loo_error_count': len(loo_errors),
    'python_version': platform.python_version(),
}

print('AI/ML-intro snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
